This notebook is adapted from [Dataflowr Module's 2 online ressource](https://github.com/dataflowr/notebooks/tree/master/Module2).

# PyTorch tensors 101

The goal of this notebook is to manipulate [PyTorch](https://docs.pytorch.org/docs/stable/index.html) that allows to handle ```tensors```, which are used to encode the signal to process, but also the internal states and parameters of models.

Key ideas for PyTorch tensors are:
- Creation (`tensor`, `zeros`, `ones`, `arange`, `linspace`, `randn`, `eye`, `empty`)
- Attributes (`dtype`, `shape`, `device`)
- Indexing/slicing, reshaping (`view`, `reshape`, `unsqueeze`, `squeeze`)
- Broadcasting
- In-place operations
- Numpy bridge
- Moving tensors to GPUs (`.to(device)`, `.cpu()`, `.cuda()`)

In [219]:
import matplotlib.pyplot as plt     # For plotting
%matplotlib inline
import torch                        # For tensor computations and deep learning
import numpy as np                  # For numerical operations on arrays

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# If you want to install PyTorch: visit the bottom page of the website https://pytorch.org.

PyTorch version: 2.0.1
Using device: cpu


### Creation and attributes

> **Q1.** Using the [`empty`](https://docs.pytorch.org/docs/stable/generated/torch.empty.html) function from Torch, construct an uninitialized 3x5 matrix. Check its default datatype and shape using the [`dtype`](https://docs.pytorch.org/docs/stable/tensor_attributes.html) and `shape` attributes, and print the values of the tensors. Comment.

In [223]:
# YOUR CODE HERE
x = torch.empty(3,5)
print(x.dtype)
print(x.shape)
print(x)

torch.float32
torch.Size([3, 5])
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


> **Q2.** Similarly, create a tensor of size 3x5 containing random values from a Gaussian $\mathcal{N}(0, 1)$. To do so, have a look at the [`randn`](https://docs.pytorch.org/docs/main/generated/torch.randn.html) function. Run the cell multiple times. Comment.

In [ ]:
# YOUR CODE HERE
x = torch.randn(3,5)
print(x)

tensor([[ 1.0587, -0.3664,  0.6827, -0.6706,  0.5570],
        [ 0.9778,  0.8663,  0.0502,  1.0002,  0.1818],
        [ 0.0374,  1.7699, -2.1751,  0.6636, -0.3653]])


One way to enforce reproducibility when dealing with randomness in PyTorch is fixing the seed. This can be achieved using the `torch.manual_seed()` function of a generator.

In [222]:
g = torch.Generator().manual_seed(42)
x = torch.randn(3, 5, generator=g)
print(x)

tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229],
        [-0.1863,  2.2082, -0.6380,  0.4617,  0.2674],
        [ 0.5349,  0.8094,  1.1103, -1.6898, -0.9890]])


Here are some popular functions to initialize torch tensors:

In [104]:
z = torch.zeros((2, 3, 3))
o = torch.ones((2, 3, 3))
e = torch.eye(3, 3)
l = torch.linspace(-1, 1, 11)
a = torch.arange(-1, 1, 0.2)

print("zeros:\n", z)
print("ones:\n", o)
print("eye:\n", e)
print("linspace:\n", l)
print("arange:\n", a)

zeros:
 tensor([[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]])
ones:
 tensor([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]])
eye:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
linspace:
 tensor([-1.0000, -0.8000, -0.6000, -0.4000, -0.2000,  0.0000,  0.2000,  0.4000,
         0.6000,  0.8000,  1.0000])
arange:
 tensor([-1.0000, -0.8000, -0.6000, -0.4000, -0.2000,  0.0000,  0.2000,  0.4000,
         0.6000,  0.8000])


### Indexing, reshaping, and slicing

To access an element of a tensor, we can simply specify the index in each dimension (starting from 0). For instance `x[2, 3]` access the element in the 3rd row, 4th column of $x$ and gives it back in a scalar-tensor.

In [225]:
x = torch.randn(4, 4)
print(x)
print(x[2, 3])
print(x[2][3])      # x[2][3] is equivalent to x[2, 3]

tensor([[ 1.7480,  0.2241, -0.0651, -0.2580],
        [ 0.4122,  0.2043, -1.0966,  1.2616],
        [ 1.2348,  1.7875,  0.9413, -1.5924],
        [ 1.6703, -0.7535,  0.1804, -0.1471]])
tensor(-1.5924)
tensor(-1.5924)


In [226]:
a = torch.arange(24)
print(a)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])


> **Q3.** $a$ is a tensor of size (24). [`Reshape`](https://docs.pytorch.org/docs/stable/generated/torch.reshape.html) it into a 2x3x4 tensor and check its shape. Note that reshape may copy the tensor in memory.

In [227]:
# YOUR CODE HERE
b = a.reshape(2, 3, 4)
print("b shape:", b.shape)
print(b)

b shape: torch.Size([2, 3, 4])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


Another way to reshape a tensor is using [`view`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html#torch.Tensor.view) function.

In [231]:
b2 = a.view(2, 3, 4)
print(b2)       # Same as b

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


However, it requires the tensor to be contiguous in memory. After some operations (like transpose or permute), the tensor is often non-contiguous so `view` will fail.

In [234]:
y = b2.T     # likely non-contiguous
y2 = y.view(24)             # .view fails

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

> **Q4.** Slice the tensor $a$ to get a new one of shape (3, 2) containing the values: `[[1, 2], [5, 6], [9, 10]]`.

In [235]:
# YOUR CODE
c = b[0, :, 1:3]
print("b[0, :, 1:3] =\n", c)

b[0, :, 1:3] =
 tensor([[ 1,  2],
        [ 5,  6],
        [ 9, 10]])


### [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)

Broadcasting automagically expands dimensions by replicating coefficients, when it is necessary to perform operations.

1. If one of the tensors has fewer dimensions than the other, it is reshaped by adding as many dimensions of size 1 as necessary in the front; then
2. For every mismatch, if one of the two tensor is of size one, it is expanded along this axis by replicating  coefficients.

If there is a tensor size mismatch for one of the dimension and neither of them is one, the operation fails.

In [237]:
v = torch.tensor([1.0, 2.0, 3.0])

> **Q5.** Create a random matrix of size 5x3. Then add $v$ to it. What happens to the resulting tensor? Explain.

In [ ]:
# YOUR CODE
M = torch.randn(5, 3)
Mpv = M + v
print(Mpv)

tensor([[1.1200, 2.4687, 1.2778],
        [0.6613, 2.0247, 2.2213],
        [2.4605, 2.2270, 1.9034],
        [2.3251, 3.1991, 2.0690],
        [0.8356, 3.0752, 3.7755]])


> **Q6.** Similarly, explain how the broadcasting worked in the code below.

In [240]:
A = torch.arange(1, 5).unsqueeze(1)
print(A.shape)
B = torch.tensor([[5., -5., 5., -5., 5.]])
print(B.shape)
C = A + B
print(C)

torch.Size([4, 1])
torch.Size([1, 5])
tensor([[ 6., -4.,  6., -4.,  6.],
        [ 7., -3.,  7., -3.,  7.],
        [ 8., -2.,  8., -2.,  8.],
        [ 9., -1.,  9., -1.,  9.]])


**Solution:**

The original (column-)vector
$$
A = \left( \begin{array}{c}
1\\
2\\
3\\
4\\
\end{array}\right)
$$
is transformed into the matrix 
$$
A = \left( \begin{array}{ccccc}
1&1&1&1&1\\
2&2&2&2&2\\
3&3&3&3&3\\
4&4&4&4&4
\end{array}\right)
$$
and the original (row-)vector
$$
B = (5,-5,5,-5,5)
$$
is transformed into the matrix
$$
B = \left( \begin{array}{ccccc}
5&-5&5&-5&5\\
5&-5&5&-5&5\\
5&-5&5&-5&5\\
5&-5&5&-5&5
\end{array}\right)
$$
so that summing these matrices gives:
$$
A+B = \left( \begin{array}{ccccc}
6&-4&6&-4&6\\
7&-3&7&-3&7\\
8&-2&8&-2&8\\
9&-1&9&-1&9
\end{array}\right)
$$

### In-place modifications

In-place operations directly modify the content of a tensor. These operations have a suffix `_`. For example: `x.copy_(y)`, `x.add_(y)`, `x.t_()`, `x.fill_(y)` will all change `x`. They can help saving some memory but can cause problems when computing gradients.

In [181]:
x = torch.randn(5, 3)
b = torch.ones(5, 3)

print(x+b)
print(x)

tensor([[ 1.4085,  1.1863,  0.9141],
        [ 1.6269,  0.4769,  1.0846],
        [ 2.4257,  1.4052,  1.5563],
        [ 1.6028, -0.5008,  0.6307],
        [ 0.3892,  2.0336,  0.6133]])
tensor([[ 0.4085,  0.1863, -0.0859],
        [ 0.6269, -0.5231,  0.0846],
        [ 1.4257,  0.4052,  0.5563],
        [ 0.6028, -1.5008, -0.3693],
        [-0.6108,  1.0336, -0.3867]])


In [183]:
x.add_(b)
print(x)        # x has changed

tensor([[2.4085, 2.1863, 1.9141],
        [2.6269, 1.4769, 2.0846],
        [3.4257, 2.4052, 2.5563],
        [2.6028, 0.4992, 1.6307],
        [1.3892, 3.0336, 1.6133]])


In [186]:
x.t_()
print(x)

tensor([[2.4085, 2.6269, 3.4257, 2.6028, 1.3892],
        [2.1863, 1.4769, 2.4052, 0.4992, 3.0336],
        [1.9141, 2.0846, 2.5563, 1.6307, 1.6133]])


### Bridge to numpy

Torch tensors can be converted to `numpy.ndarray` using the [`torch.Tensor.numpy`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html) function which can be used as

In [167]:
x = torch.randn(5, 3)
y = x.numpy()
print(y)
print(type(y))

[[-0.86732364  2.373101   -1.2504785 ]
 [-0.98764765  0.3903675   0.956884  ]
 [-0.55814266 -0.72611177 -0.05098372]
 [-0.6165619  -0.37017247  1.1566299 ]
 [-0.22449636  0.4976532  -0.17672361]]
<class 'numpy.ndarray'>


Similarly, a `numpy.ndarray` can be transformed into a `torch.tensor`using the [`torch.from_numpy()`](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html) function.

In [168]:
a = np.ones(5)
b = torch.from_numpy(a)
print(a.dtype)
print(b)

float64
tensor([1., 1., 1., 1., 1.], dtype=torch.float64)


In [169]:
xr = torch.randn(3, 5)
print(xr.dtype, xr)

torch.float32 tensor([[ 1.5455, -0.3367, -2.1127,  0.4583, -2.1671],
        [ 0.5075,  0.7953, -0.9676,  1.1404, -0.3907],
        [ 0.0233, -0.7705,  0.8573, -0.8830, -0.0673]])


Note: Be careful with types!

In [173]:
b1 = torch.ones(5, dtype=torch.float64)
b2 = torch.ones(5, dtype=torch.long)
r = torch.randn(3, 5, generator=torch.Generator().manual_seed(42))
x1 = r + b1
x2 = r + b2

print(x1)
print(x2)
print(b1)
print(b2)
print(x1 == x2)     # Falses because different dtypes

tensor([[ 1.3367,  1.1288,  1.2345,  1.2303, -0.1229],
        [ 0.8137,  3.2082,  0.3620,  1.4617,  1.2674],
        [ 1.5349,  1.8094,  2.1103, -0.6898,  0.0110]], dtype=torch.float64)
tensor([[ 1.3367,  1.1288,  1.2345,  1.2303, -0.1229],
        [ 0.8137,  3.2082,  0.3620,  1.4617,  1.2674],
        [ 1.5349,  1.8094,  2.1103, -0.6898,  0.0110]])
tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
tensor([1, 1, 1, 1, 1])
tensor([[False, False, False, False,  True],
        [ True,  True,  True, False, False],
        [False, False,  True,  True,  True]])


### Shared memory

Also be careful, changing the torch tensor modify the numpy array and vice-versa...

This is explained in the PyTorch documentation [here](https://pytorch.org/docs/stable/torch.html#torch.from_numpy):
The returned tensor by `torch.from_numpy` and ndarray share the same memory. Modifications to the tensor will be reflected in the ndarray and vice versa. 

In [187]:
a = np.ones(5)
b = torch.from_numpy(a)
print(b)

tensor([1., 1., 1., 1., 1.], dtype=torch.float64)


In [188]:
a[2] = 0
print(b)    # b has changed while we only modified a

tensor([1., 1., 0., 1., 1.], dtype=torch.float64)


In [189]:
b[3] = 5
print(a)    # a has changed while we only modified b

[1. 1. 0. 5. 1.]


### Devices and cuda

Tensors can be loaded either on CPUs or on GPUs. To check whether GPUs are available, we can call `torch.cuda.is_available()` returning a boolean.

In [191]:
torch.cuda.is_available()

False

In [209]:
device = torch.device('cpu')   # Use this to run on CPU
# device = torch.device('cuda')   # Use this to run on GPU

In [210]:
x = torch.ones(10)
print(x.device)  # Should show "cpu" (default device)

cpu


To load something on a specific device, we should create the tensor and precise the device argument.

In [211]:
x = torch.ones(10, device=device)
print(x.device)  # Should show "cpu" or "cuda" depending on the device

cpu


In [212]:
# let us run this cell only if CUDA is available
# We will use ``torch.device`` objects to move tensors in and out of GPU
if torch.cuda.is_available():
    y = torch.ones_like(x, device=device)  # directly create a tensor on GPU
    x = x.to(device)                       # or just use strings ``.to("cuda")``
    z = x + y
    print(z,z.type())
    print(z.to("cpu", torch.double))       # ``.to`` can also change dtype together!

In [213]:
x = torch.randn(1)
x = x.to(device)

In [214]:
x.device

device(type='cpu')

In [216]:
# the following line is only useful if CUDA is available
x = x.data
print(x)
print(x.item())
print(x.cpu().numpy())  # move to CPU first, then convert to numpy

tensor([1.3871])
1.387088656425476
[1.3870887]


## Simple interfaces to standard image data-bases

An example with the [CIFAR10](https://pytorch.org/docs/stable/torchvision/datasets.html#torchvision.datasets.CIFAR10) dataset.

In [ ]:
import torchvision

data_dir = 'content/data'

cifar = torchvision.datasets.CIFAR10(data_dir, train = True, download = True)
cifar.data.shape

Documentation about the [`permute`](https://pytorch.org/docs/stable/tensors.html#torch.Tensor.permute) operation.

In [ ]:
x = torch.from_numpy(cifar.data).permute(0,3,1,2).float()
x = x / 255
print(x.type(), x.size(), x.min().item(), x.max().item())

Documentation about the [`narrow(input, dim, start, length)`](https://pytorch.org/docs/stable/torch.html#torch.narrow) operation.

In [ ]:
# Narrows to the first images, converts to float
x = torch.narrow(x, 0, 0, 48)

In [ ]:
x.shape

In [ ]:
# Showing images
def show(img):
    npimg = img.numpy()
    plt.figure(figsize=(20,10))
    plt.imshow(np.transpose(npimg, (1,2,0)), interpolation='nearest')
    
show(torchvision.utils.make_grid(x, nrow = 12))

In [ ]:
# Kills the green and blue channels
x.narrow(1, 1, 2).fill_(0)
show(torchvision.utils.make_grid(x, nrow = 12))